# 07 — Validation, Reliability, and Reproducibility Checks

This notebook creates transparent audit evidence for the Dutch PEP benchmark and the OpenSanctions matching analysis.

It is designed to support claims in the methodology, results, and limitations chapters. It does **not** prove that the benchmark or OpenSanctions is complete, and it does not turn fuzzy name similarity into an automatic match.

## What this notebook does

1. checks the integrity and traceability of the final benchmark and matching outputs;
2. produces benchmark-quality exception lists for manual follow-up;
3. creates reproducible, stratified review templates for strict matches, rule-based matches, and fuzzy candidates;
4. creates a separate repeat-review template for an intra-coder consistency check;
5. reports conservative coverage sensitivity scenarios without changing the primary coverage estimate; and
6. records input-file fingerprints, environment information, parameters, and pipeline row counts.

## Required run order

Run notebooks **02** and **03** from top to bottom first. This notebook can be run after notebooks 04–06, but does not depend on them.

## Important interpretation rules

- The primary result remains the **strict confirmed coverage** from notebook 03.
- Fuzzy candidates are review aids only. They are never automatically added to coverage.
- The strict-match sample is an **audit sample**, not a statistical estimate of match accuracy.
- The repeat review is an **intra-coder consistency check**. It is not inter-rater reliability because the same reviewer completes both rounds.
- Any unresolved quality exception should be reported transparently as a limitation rather than silently removed.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import platform
import sys

import numpy as np
import pandas as pd
from IPython.display import display

# -------------------------------------------------------------------
# Project folders
# -------------------------------------------------------------------

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

INPUT_DIR = PROJECT_DIR / "data" / "input"
EXTERNAL_DIR = PROJECT_DIR / "data" / "external"
OUTPUT_DIR = PROJECT_DIR / "data" / "output"
VALIDATION_DIR = OUTPUT_DIR / "validation_checks"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------
# Inputs created by earlier notebooks
# -------------------------------------------------------------------

BENCHMARK_PATH = OUTPUT_DIR / "main_current_benchmark_v3_extended.csv"
MATCH_RESULTS_PATH = OUTPUT_DIR / "benchmark_v3_opensanctions_match_results.csv"
MATCH_SUMMARY_PATH = OUTPUT_DIR / "benchmark_v3_opensanctions_match_summary.csv"
FUZZY_CANDIDATES_PATH = OUTPUT_DIR / "benchmark_v3_opensanctions_fuzzy_candidates.csv"
RULE_BASED_MATCHES_PATH = OUTPUT_DIR / "benchmark_v3_opensanctions_initial_surname_matches.csv"
RAW_COMPONENTS_PATH = OUTPUT_DIR / "benchmark_v3_components_before_cleaning.csv"
OPEN_SANCTIONS_PATH = EXTERNAL_DIR / "opensanctions_targets.simple.csv"

# -------------------------------------------------------------------
# Reproducible review settings
# -------------------------------------------------------------------

RANDOM_SEED = 42
STRICT_AUDIT_SAMPLE_TARGET = 30
FUZZY_AUDIT_SAMPLE_TARGET = 20
REPEAT_REVIEW_SAMPLE_TARGET = 10

# -------------------------------------------------------------------
# Validation outputs
# -------------------------------------------------------------------

DATASET_INTEGRITY_PATH = VALIDATION_DIR / "dataset_integrity_checks.csv"
BENCHMARK_QUALITY_PATH = VALIDATION_DIR / "benchmark_quality_checks.csv"
BENCHMARK_EXCEPTIONS_PATH = VALIDATION_DIR / "benchmark_quality_exceptions.csv"
CATEGORY_TRACEABILITY_PATH = VALIDATION_DIR / "category_traceability_profile.csv"
MATCHING_INTEGRITY_PATH = VALIDATION_DIR / "matching_integrity_checks.csv"
ROW_COUNTS_PATH = VALIDATION_DIR / "pipeline_row_counts.csv"
SENSITIVITY_PATH = VALIDATION_DIR / "pre_review_candidate_sensitivity_diagnostic.csv"
POST_REVIEW_SENSITIVITY_PATH = VALIDATION_DIR / "post_review_sensitivity_analysis.csv"
POST_REVIEW_AUDIT_PATH = VALIDATION_DIR / "post_review_matching_audit_summary.csv"
FUZZY_THRESHOLD_PATH = VALIDATION_DIR / "fuzzy_candidate_threshold_diagnostic.csv"
MANIFEST_PATH = VALIDATION_DIR / "reproducibility_file_manifest.csv"
ENVIRONMENT_PATH = VALIDATION_DIR / "python_environment_record.csv"
RUN_METADATA_PATH = VALIDATION_DIR / "validation_run_metadata.json"

STRICT_REVIEW_PATH = VALIDATION_DIR / "strict_match_manual_review_sample.csv"
RULE_REVIEW_PATH = VALIDATION_DIR / "rule_based_match_manual_review.csv"
FUZZY_REVIEW_PATH = VALIDATION_DIR / "fuzzy_candidate_manual_review_sample.csv"
REVIEW_SUMMARY_PATH = VALIDATION_DIR / "manual_review_summary.csv"
MATCH_FINDINGS_PATH = VALIDATION_DIR / "matching_quality_findings.csv"

REPEAT_REVIEW_PATH = VALIDATION_DIR / "intra_coder_repeat_review_template.csv"
INTRA_CODER_SUMMARY_PATH = VALIDATION_DIR / "intra_coder_consistency_summary.csv"
INTRA_CODER_DISAGREEMENTS_PATH = VALIDATION_DIR / "intra_coder_disagreements.csv"

print("Project folder:", PROJECT_DIR)
print("Validation output folder:", VALIDATION_DIR)
print("\nCore analysis files:")
for label, file_path in {
    "Final benchmark": BENCHMARK_PATH,
    "Final match results": MATCH_RESULTS_PATH,
    "Match summary": MATCH_SUMMARY_PATH,
    "Fuzzy candidates": FUZZY_CANDIDATES_PATH,
    "Rule-based matches": RULE_BASED_MATCHES_PATH,
}.items():
    print(f"- {label}: {file_path.exists()}  |  {file_path}")

## 1. Helper functions and input loading

The helper functions below make the notebook robust to common Excel encodings and preserve completed manual-review fields if the notebook is rerun.

Do not edit the original benchmark or matching files in this notebook. All validation outputs are written to `data/output/validation_checks/`.

In [ ]:
# -------------------------------------------------------------------
# General helpers
# -------------------------------------------------------------------

def read_csv_with_fallback_encodings(file_path: Path, required: bool = True) -> pd.DataFrame:
    """Read a CSV using common encodings; return an empty table for optional empty files."""
    if not file_path.exists():
        if required:
            raise FileNotFoundError(
                "Required analysis file was not found:\n"
                f"{file_path}\n\n"
                "Run the required preceding notebook(s) first."
            )
        return pd.DataFrame()

    for encoding in ["utf-8-sig", "utf-8", "cp1252", "latin1"]:
        try:
            return pd.read_csv(file_path, encoding=encoding)
        except UnicodeDecodeError:
            continue
        except pd.errors.EmptyDataError:
            return pd.DataFrame()

    raise ValueError(f"Could not read {file_path.name} with the supported encodings.")


def write_csv_safely(dataframe: pd.DataFrame, output_path: Path) -> Path:
    """Save CSV output and use a timestamped fallback if the target is open in Excel."""
    try:
        dataframe.to_csv(output_path, index=False, encoding="utf-8-sig")
        return output_path
    except PermissionError:
        timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
        fallback_path = output_path.with_name(
            f"{output_path.stem}_{timestamp}{output_path.suffix}"
        )
        dataframe.to_csv(fallback_path, index=False, encoding="utf-8-sig")
        print(
            f"Warning: {output_path.name} appears to be open. "
            f"Saved a fallback file: {fallback_path.name}"
        )
        return fallback_path


def blank_or_missing_mask(series: pd.Series) -> pd.Series:
    """Return True for missing or blank values."""
    return (
        series.isna()
        | series.astype("string").str.strip().eq("")
    ).fillna(True)


def as_bool(series: pd.Series) -> pd.Series:
    """Convert common boolean encodings to a Boolean Series."""
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes", "y"])
    )


def require_columns(dataframe: pd.DataFrame, required_columns: list[str], label: str) -> None:
    """Stop early with an actionable message when a core file has the wrong schema."""
    missing_columns = [
        column for column in required_columns
        if column not in dataframe.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{label} is missing the following required columns:\n"
            f"{missing_columns}\n\n"
            "Rerun the relevant earlier notebook from a fresh kernel."
        )


def add_missing_columns(dataframe: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """Add empty columns to optional audit files that may contain zero rows."""
    dataframe = dataframe.copy()

    for column in columns:
        if column not in dataframe.columns:
            dataframe[column] = pd.NA

    return dataframe


def file_sha256(file_path: Path) -> str:
    """Create a stable SHA-256 fingerprint without loading the whole file into memory."""
    digest = hashlib.sha256()

    with open(file_path, "rb") as file_handle:
        for block in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()


def package_version(package_name: str) -> str:
    """Return installed package version or a transparent 'not installed' value."""
    try:
        return importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"


def preserve_existing_review_fields(
    new_template_df: pd.DataFrame,
    existing_path: Path,
    key_column: str,
    review_columns: list[str]
) -> pd.DataFrame:
    """
    Preserve completed review fields by stable case identifier.

    This allows the notebook to be rerun without overwriting decisions entered
    in Excel. Newly selected cases are added with blank review fields.
    """
    new_template_df = new_template_df.copy()

    for column in review_columns:
        if column not in new_template_df.columns:
            new_template_df[column] = pd.NA

    if not existing_path.exists():
        return new_template_df

    existing_df = read_csv_with_fallback_encodings(existing_path, required=False)

    if existing_df.empty or key_column not in existing_df.columns:
        return new_template_df

    keep_columns = [
        key_column,
        *[
            column
            for column in review_columns
            if column in existing_df.columns
        ]
    ]

    existing_review_df = (
        existing_df[keep_columns]
        .drop_duplicates(subset=[key_column], keep="last")
        .copy()
    )

    merged_df = new_template_df.drop(
        columns=review_columns,
        errors="ignore"
    ).merge(
        existing_review_df,
        on=key_column,
        how="left"
    )

    for column in review_columns:
        if column not in merged_df.columns:
            merged_df[column] = pd.NA

    return merged_df


def stratified_sample(
    dataframe: pd.DataFrame,
    stratum_column: str,
    target_n: int,
    random_seed: int
) -> pd.DataFrame:
    """
    Draw a transparent, reproducible sample with at least one case per
    available stratum where the target size allows it.

    Remaining places are allocated approximately in proportion to stratum size.
    This is an audit-sampling approach, not a probability estimate.
    """
    if dataframe.empty or target_n <= 0:
        return dataframe.head(0).copy()

    working_df = dataframe.copy()
    working_df[stratum_column] = (
        working_df[stratum_column]
        .astype("string")
        .fillna("missing_stratum")
    )

    stratum_counts = (
        working_df.groupby(stratum_column, dropna=False)
        .size()
        .sort_values(ascending=False)
    )

    target_n = min(target_n, len(working_df))

    if len(stratum_counts) > target_n:
        selected_strata = stratum_counts.head(target_n).index.tolist()
        allocation = {stratum: 1 for stratum in selected_strata}
    else:
        allocation = {stratum: 1 for stratum in stratum_counts.index.tolist()}
        remaining = target_n - len(allocation)

        capacities = {
            stratum: int(stratum_counts.loc[stratum] - allocation[stratum])
            for stratum in allocation
        }

        while remaining > 0 and sum(capacities.values()) > 0:
            eligible_strata = [
                stratum
                for stratum, capacity in capacities.items()
                if capacity > 0
            ]

            chosen_stratum = max(
                eligible_strata,
                key=lambda stratum: (
                    capacities[stratum] / stratum_counts.loc[stratum],
                    stratum_counts.loc[stratum]
                )
            )

            allocation[chosen_stratum] += 1
            capacities[chosen_stratum] -= 1
            remaining -= 1

    sampled_parts = []

    for number, (stratum, allocated_n) in enumerate(sorted(allocation.items())):
        stratum_df = working_df.loc[
            working_df[stratum_column].eq(stratum)
        ].copy()

        sampled_parts.append(
            stratum_df.sample(
                n=min(allocated_n, len(stratum_df)),
                random_state=random_seed + number
            )
        )

    return (
        pd.concat(sampled_parts, ignore_index=True)
        .sort_values([stratum_column], kind="stable")
        .reset_index(drop=True)
    )


# -------------------------------------------------------------------
# Load the core analysis outputs
# -------------------------------------------------------------------

benchmark_df = read_csv_with_fallback_encodings(BENCHMARK_PATH)
match_df = read_csv_with_fallback_encodings(MATCH_RESULTS_PATH)
match_summary_df = read_csv_with_fallback_encodings(MATCH_SUMMARY_PATH)
fuzzy_candidates_df = read_csv_with_fallback_encodings(
    FUZZY_CANDIDATES_PATH,
    required=False
)
rule_based_matches_df = read_csv_with_fallback_encodings(
    RULE_BASED_MATCHES_PATH,
    required=False
)

# Optional historical-stage output. It is used only for row-count reporting.
raw_components_df = read_csv_with_fallback_encodings(
    RAW_COMPONENTS_PATH,
    required=False
)

require_columns(
    benchmark_df,
    [
        "benchmark_record_id",
        "person_name",
        "normalised_name",
        "taxonomy_category",
        "role_title",
        "institution",
        "source_url",
        "source_component",
        "access_timestamp_utc",
        "included_in_main_benchmark"
    ],
    "Final benchmark"
)

require_columns(
    match_df,
    [
        "benchmark_record_id",
        "person_name",
        "taxonomy_category",
        "match_type",
        "strict_confirmed_match",
        "rule_based_probable_match",
        "included_in_extended_coverage",
        "final_match_status"
    ],
    "Final matching results"
)

# Ensure optional audit tables have usable schemas even if no records were created.
fuzzy_candidates_df = add_missing_columns(
    fuzzy_candidates_df,
    [
        "benchmark_record_id",
        "person_name",
        "taxonomy_category",
        "role_title",
        "institution",
        "candidate_rank",
        "candidate_score",
        "same_surname",
        "first_initial_matches",
        "candidate_priority",
        "candidate_opensanctions_id",
        "candidate_opensanctions_name",
        "candidate_dataset"
    ]
)

rule_based_matches_df = add_missing_columns(
    rule_based_matches_df,
    [
        "benchmark_record_id",
        "person_name",
        "taxonomy_category",
        "role_title",
        "institution",
        "opensanctions_id",
        "opensanctions_name",
        "dataset",
        "match_type",
        "rule_description"
    ]
)

print("Loaded row counts:")
print(f"- Final benchmark: {len(benchmark_df):,}")
print(f"- Final match results: {len(match_df):,}")
print(f"- Fuzzy candidate rows: {len(fuzzy_candidates_df):,}")
print(f"- Rule-based match rows: {len(rule_based_matches_df):,}")

## 2. Dataset-integrity and benchmark-quality checks

The following checks verify whether the final benchmark is internally consistent and whether each final record retains the minimum provenance needed for audit.

The notebook reports exceptions rather than editing source data. A `review` status means that the record should either be completed, explained, or transparently discussed as a limitation.

In [ ]:
# -------------------------------------------------------------------
# Dataset-integrity checks
# -------------------------------------------------------------------

integrity_rows = []

def add_integrity_check(check_name: str, result, expected, status: str, details: str) -> None:
    integrity_rows.append({
        "check_name": check_name,
        "result": result,
        "expected": expected,
        "status": status,
        "details": details
    })


benchmark_ids = benchmark_df["benchmark_record_id"].astype("string")
match_ids = match_df["benchmark_record_id"].astype("string")

benchmark_duplicate_ids = int(benchmark_ids.duplicated(keep=False).sum())
match_duplicate_ids = int(match_ids.duplicated(keep=False).sum())

add_integrity_check(
    "benchmark_record_id_present",
    int(blank_or_missing_mask(benchmark_df["benchmark_record_id"]).sum()),
    0,
    "pass" if blank_or_missing_mask(benchmark_df["benchmark_record_id"]).sum() == 0 else "review",
    "Every benchmark record must have a stable identifier."
)

add_integrity_check(
    "benchmark_record_id_unique",
    benchmark_duplicate_ids,
    0,
    "pass" if benchmark_duplicate_ids == 0 else "review",
    "The final benchmark should contain one row per benchmark_record_id."
)

add_integrity_check(
    "match_results_record_id_unique",
    match_duplicate_ids,
    0,
    "pass" if match_duplicate_ids == 0 else "review",
    "The matching output should contain one final row per benchmark record."
)

add_integrity_check(
    "benchmark_and_match_row_count_reconcile",
    f"{len(benchmark_df)} benchmark / {len(match_df)} match rows",
    "equal row counts",
    "pass" if len(benchmark_df) == len(match_df) else "review",
    "Matching results should cover every final benchmark record exactly once."
)

missing_from_match = sorted(set(benchmark_ids) - set(match_ids))
extra_in_match = sorted(set(match_ids) - set(benchmark_ids))

add_integrity_check(
    "all_benchmark_ids_present_in_match_results",
    len(missing_from_match),
    0,
    "pass" if len(missing_from_match) == 0 else "review",
    "No final benchmark record should be absent from the matching output."
)

add_integrity_check(
    "no_extra_match_result_ids",
    len(extra_in_match),
    0,
    "pass" if len(extra_in_match) == 0 else "review",
    "The matching output should not contain records outside the final benchmark."
)

benchmark_categories = set(
    benchmark_df["taxonomy_category"]
    .dropna()
    .astype(str)
)
match_categories = set(
    match_df["taxonomy_category"]
    .dropna()
    .astype(str)
)

add_integrity_check(
    "taxonomy_categories_reconcile",
    f"{len(benchmark_categories)} benchmark / {len(match_categories)} matching",
    "identical category sets",
    "pass" if benchmark_categories == match_categories else "review",
    "Coverage must be calculated against the same taxonomy categories used in the benchmark."
)

required_benchmark_columns = [
    "benchmark_record_id",
    "person_name",
    "normalised_name",
    "taxonomy_category",
    "role_title",
    "institution",
    "source_url",
    "source_component",
    "access_timestamp_utc",
    "included_in_main_benchmark"
]

for column_name in required_benchmark_columns:
    add_integrity_check(
        f"required_column_present__{column_name}",
        int(column_name in benchmark_df.columns),
        1,
        "pass" if column_name in benchmark_df.columns else "review",
        "Required benchmark field is available for traceability or matching."
    )

dataset_integrity_df = pd.DataFrame(integrity_rows)
write_csv_safely(dataset_integrity_df, DATASET_INTEGRITY_PATH)

display(dataset_integrity_df)

print("\nIntegrity checks requiring review:")
display(
    dataset_integrity_df.loc[
        dataset_integrity_df["status"].ne("pass")
    ]
)

In [ ]:
# -------------------------------------------------------------------
# Benchmark-quality checks and exception register
# -------------------------------------------------------------------

quality_rows = []
exception_frames = []

def add_quality_check(
    check_name: str,
    mask: pd.Series,
    definition: str
) -> None:
    mask = mask.fillna(True)
    exception_count = int(mask.sum())

    quality_rows.append({
        "check_name": check_name,
        "records_requiring_review": exception_count,
        "status": "pass" if exception_count == 0 else "review",
        "definition": definition
    })

    if exception_count > 0:
        exception_columns = [
            column
            for column in [
                "benchmark_record_id",
                "person_name",
                "taxonomy_category",
                "role_title",
                "institution",
                "source_url",
                "source_component",
                "access_timestamp_utc",
                "included_in_main_benchmark"
            ]
            if column in benchmark_df.columns
        ]

        exception_df = benchmark_df.loc[
            mask,
            exception_columns
        ].copy()

        exception_df.insert(0, "quality_issue", check_name)
        exception_frames.append(exception_df)


included_positive_values = {"yes", "true", "1", "y"}

included_decision_missing_mask = blank_or_missing_mask(
    benchmark_df["included_in_main_benchmark"]
)

included_decision_not_positive_mask = (
    ~benchmark_df["included_in_main_benchmark"]
    .astype("string")
    .str.strip()
    .str.lower()
    .isin(included_positive_values)
)

source_url_missing_mask = blank_or_missing_mask(benchmark_df["source_url"])
source_component_missing_mask = blank_or_missing_mask(
    benchmark_df["source_component"]
)
timestamp_missing_mask = blank_or_missing_mask(
    benchmark_df["access_timestamp_utc"]
)
taxonomy_missing_mask = blank_or_missing_mask(
    benchmark_df["taxonomy_category"]
)
name_missing_mask = blank_or_missing_mask(benchmark_df["person_name"])
normalised_name_missing_mask = blank_or_missing_mask(
    benchmark_df["normalised_name"]
)
role_missing_mask = blank_or_missing_mask(benchmark_df["role_title"])
institution_missing_mask = blank_or_missing_mask(
    benchmark_df["institution"]
)

# "Sufficient identifying information" is a project-specific operational check:
# it does not prove identity, but it confirms that a reviewer has a name, role,
# institution, category, and source context available.
identifying_information_incomplete_mask = (
    name_missing_mask
    | normalised_name_missing_mask
    | taxonomy_missing_mask
    | role_missing_mask
    | institution_missing_mask
    | source_url_missing_mask
)

source_traceability_incomplete_mask = (
    source_url_missing_mask
    | source_component_missing_mask
    | timestamp_missing_mask
)

duplicate_name_category_mask = benchmark_df.duplicated(
    subset=["normalised_name", "taxonomy_category"],
    keep=False
)

add_quality_check(
    "missing_inclusion_decision",
    included_decision_missing_mask,
    "Every final record should retain an explicit inclusion decision."
)

add_quality_check(
    "included_record_not_marked_positive",
    included_decision_not_positive_mask,
    "The final benchmark should contain only records marked as included."
)

add_quality_check(
    "missing_source_url",
    source_url_missing_mask,
    "Every included record should retain a source URL or source reference."
)

add_quality_check(
    "missing_source_component",
    source_component_missing_mask,
    "Every included record should retain the collection component that produced it."
)

add_quality_check(
    "missing_access_timestamp_utc",
    timestamp_missing_mask,
    "Every included record should retain an extraction or access timestamp."
)

add_quality_check(
    "missing_taxonomy_category",
    taxonomy_missing_mask,
    "Every included record should be assigned to one taxonomy category."
)

add_quality_check(
    "insufficient_identifying_information",
    identifying_information_incomplete_mask,
    "Name, normalised name, role, institution, taxonomy category, and source URL are required for auditability."
)

add_quality_check(
    "incomplete_source_traceability",
    source_traceability_incomplete_mask,
    "Source URL, source component, and access timestamp should be available together."
)

add_quality_check(
    "duplicate_normalised_name_within_taxonomy_category",
    duplicate_name_category_mask,
    "The final benchmark should retain one record per normalised person name and taxonomy category."
)

benchmark_quality_df = pd.DataFrame(quality_rows)

if exception_frames:
    benchmark_exceptions_df = (
        pd.concat(exception_frames, ignore_index=True)
        .sort_values(
            ["quality_issue", "taxonomy_category", "person_name"],
            kind="stable"
        )
        .reset_index(drop=True)
    )
else:
    benchmark_exceptions_df = pd.DataFrame(
        columns=[
            "quality_issue",
            "benchmark_record_id",
            "person_name",
            "taxonomy_category"
        ]
    )

write_csv_safely(benchmark_quality_df, BENCHMARK_QUALITY_PATH)
write_csv_safely(benchmark_exceptions_df, BENCHMARK_EXCEPTIONS_PATH)

display(benchmark_quality_df)

print("\nBenchmark-quality exceptions:")
display(benchmark_exceptions_df.head(100))

In [ ]:
# -------------------------------------------------------------------
# Category-level traceability profile
# -------------------------------------------------------------------

traceability_df = benchmark_df.copy()

traceability_df["has_source_url"] = ~blank_or_missing_mask(
    traceability_df["source_url"]
)
traceability_df["has_source_component"] = ~blank_or_missing_mask(
    traceability_df["source_component"]
)
traceability_df["has_access_timestamp"] = ~blank_or_missing_mask(
    traceability_df["access_timestamp_utc"]
)
traceability_df["has_identifying_information"] = ~(
    blank_or_missing_mask(traceability_df["person_name"])
    | blank_or_missing_mask(traceability_df["normalised_name"])
    | blank_or_missing_mask(traceability_df["role_title"])
    | blank_or_missing_mask(traceability_df["institution"])
    | blank_or_missing_mask(traceability_df["source_url"])
)

category_traceability_df = (
    traceability_df
    .groupby("taxonomy_category", dropna=False)
    .agg(
        benchmark_records=("benchmark_record_id", "size"),
        records_with_source_url=("has_source_url", "sum"),
        records_with_source_component=("has_source_component", "sum"),
        records_with_access_timestamp=("has_access_timestamp", "sum"),
        records_with_identifying_information=(
            "has_identifying_information",
            "sum"
        ),
        source_components=(
            "source_component",
            lambda values: " | ".join(
                sorted(
                    {
                        str(value)
                        for value in values.dropna()
                        if str(value).strip() != ""
                    }
                )
            )
        )
    )
    .reset_index()
)

for column_name in [
    "records_with_source_url",
    "records_with_source_component",
    "records_with_access_timestamp",
    "records_with_identifying_information"
]:
    category_traceability_df[
        f"{column_name}_percentage"
    ] = (
        category_traceability_df[column_name]
        / category_traceability_df["benchmark_records"]
        * 100
    ).round(1)

category_traceability_df = category_traceability_df.sort_values(
    "benchmark_records",
    ascending=False
).reset_index(drop=True)

write_csv_safely(category_traceability_df, CATEGORY_TRACEABILITY_PATH)

print("Category-level traceability profile:")
display(category_traceability_df)

## 3. Matching-output checks and coverage sensitivity

These checks confirm that the final matching table remains consistent with the documented hierarchy in notebook 03.

The sensitivity table compares only transparent alternate rules:

- **strict confirmed coverage**: unique normalised and cleaned exact matches;
- **extended rule-based coverage**: strict matches plus unique initial–surname matches;
- the same measures after removing records with an exact-name collision from the denominator.

Fuzzy score thresholds are shown separately as **review-volume diagnostics**. They are not presented as coverage estimates because fuzzy similarity alone is not evidence of identity.

In [ ]:
# -------------------------------------------------------------------
# Matching-output integrity checks
# -------------------------------------------------------------------

matching_integrity_rows = []

def add_matching_check(check_name: str, result, expected, status: str, details: str) -> None:
    matching_integrity_rows.append({
        "check_name": check_name,
        "result": result,
        "expected": expected,
        "status": status,
        "details": details
    })


strict_match_types = {"normalised_exact", "cleaned_exact"}
rule_match_types = {"initial_surname_unique"}

strict_indicator = as_bool(match_df["strict_confirmed_match"])
rule_indicator = as_bool(match_df["rule_based_probable_match"])
extended_indicator = as_bool(match_df["included_in_extended_coverage"])

expected_strict_indicator = match_df["match_type"].isin(strict_match_types)
expected_rule_indicator = match_df["match_type"].isin(rule_match_types)
expected_extended_indicator = (
    expected_strict_indicator
    | expected_rule_indicator
)

strict_indicator_inconsistencies = int(
    (strict_indicator != expected_strict_indicator).sum()
)
rule_indicator_inconsistencies = int(
    (rule_indicator != expected_rule_indicator).sum()
)
extended_indicator_inconsistencies = int(
    (extended_indicator != expected_extended_indicator).sum()
)

add_matching_check(
    "strict_indicator_matches_documented_match_types",
    strict_indicator_inconsistencies,
    0,
    "pass" if strict_indicator_inconsistencies == 0 else "review",
    "Strict matches should be normalised_exact or cleaned_exact only."
)

add_matching_check(
    "rule_based_indicator_matches_documented_match_type",
    rule_indicator_inconsistencies,
    0,
    "pass" if rule_indicator_inconsistencies == 0 else "review",
    "Rule-based matches should be initial_surname_unique only."
)

add_matching_check(
    "extended_indicator_reconciles_with_strict_or_rule_based",
    extended_indicator_inconsistencies,
    0,
    "pass" if extended_indicator_inconsistencies == 0 else "review",
    "Extended coverage should equal strict confirmed OR rule-based probable match."
)

rule_ids_in_match = set(
    match_df.loc[
        match_df["match_type"].eq("initial_surname_unique"),
        "benchmark_record_id"
    ].astype("string")
)
rule_audit_ids = set(
    rule_based_matches_df["benchmark_record_id"]
    .dropna()
    .astype("string")
)

add_matching_check(
    "rule_based_audit_file_reconciles_with_final_results",
    len(rule_ids_in_match.symmetric_difference(rule_audit_ids)),
    0,
    "pass" if rule_ids_in_match == rule_audit_ids else "review",
    "Every final initial–surname record should appear in the rule-based audit file."
)

fuzzy_ids = set(
    fuzzy_candidates_df["benchmark_record_id"]
    .dropna()
    .astype("string")
)
final_ids = set(match_df["benchmark_record_id"].astype("string"))

add_matching_check(
    "fuzzy_candidate_ids_exist_in_final_results",
    len(fuzzy_ids - final_ids),
    0,
    "pass" if fuzzy_ids.issubset(final_ids) else "review",
    "Fuzzy candidates should relate only to records in the final benchmark matching output."
)

matching_integrity_df = pd.DataFrame(matching_integrity_rows)
write_csv_safely(matching_integrity_df, MATCHING_INTEGRITY_PATH)

display(matching_integrity_df)

In [ ]:
# -------------------------------------------------------------------
# Pre-review candidate diagnostic (not final coverage)
# -------------------------------------------------------------------

benchmark_total = len(match_df)
strict_count = int(strict_indicator.sum())
rule_based_count = int(rule_indicator.sum())
extended_count = int(extended_indicator.sum())

ambiguous_exact_mask = match_df["final_match_status"].eq(
    "ambiguous_exact_name_candidate"
)
ambiguous_exact_count = int(ambiguous_exact_mask.sum())
non_ambiguous_denominator = benchmark_total - ambiguous_exact_count

sensitivity_rows = [
    {
        "scenario": "primary_strict_confirmed_coverage",
        "numerator_matches": strict_count,
        "denominator_records": benchmark_total,
        "coverage_rate": strict_count / benchmark_total if benchmark_total else np.nan,
        "interpretation": (
            "Primary estimate: unique normalised exact and cleaned exact matches."
        )
    },
    {
        "scenario": "pre_review_initial_surname_candidate_measure",
        "numerator_matches": extended_count,
        "denominator_records": benchmark_total,
        "coverage_rate": extended_count / benchmark_total if benchmark_total else np.nan,
        "interpretation": (
            "Pre-review diagnostic only: strict matches plus all initial–surname candidates. Do not report as coverage after manual audit."
        )
    },
    {
        "scenario": "strict_coverage_excluding_ambiguous_exact_name_records",
        "numerator_matches": strict_count,
        "denominator_records": non_ambiguous_denominator,
        "coverage_rate": (
            strict_count / non_ambiguous_denominator
            if non_ambiguous_denominator else np.nan
        ),
        "interpretation": (
            "Pre-review diagnostic only: removes exact-name collision cases from the denominator; do not report after manual audit."
        )
    },
    {
        "scenario": "pre_review_initial_surname_candidate_measure_excluding_ambiguous_exact_names",
        "numerator_matches": extended_count,
        "denominator_records": non_ambiguous_denominator,
        "coverage_rate": (
            extended_count / non_ambiguous_denominator
            if non_ambiguous_denominator else np.nan
        ),
        "interpretation": (
            "Sensitivity only: removes exact-name collision cases from the denominator."
        )
    }
]

sensitivity_df = pd.DataFrame(sensitivity_rows)
sensitivity_df["coverage_percentage"] = (
    sensitivity_df["coverage_rate"] * 100
).round(1)

write_csv_safely(sensitivity_df, SENSITIVITY_PATH)

print("Pre-review candidate diagnostic (not final coverage):")
display(sensitivity_df)

print(
    "\nImportant: fuzzy score thresholds below are candidate-review diagnostics, "
    "not coverage estimates."
)

# -------------------------------------------------------------------
# Fuzzy score threshold diagnostic
# -------------------------------------------------------------------

fuzzy_best_df = (
    fuzzy_candidates_df
    .dropna(subset=["benchmark_record_id"])
    .sort_values(
        ["benchmark_record_id", "candidate_score", "candidate_rank"],
        ascending=[True, False, True],
        kind="stable"
    )
    .drop_duplicates(subset=["benchmark_record_id"], keep="first")
    .copy()
)

threshold_rows = []

for threshold in [85, 90, 95]:
    above_threshold = fuzzy_best_df[
        pd.to_numeric(
            fuzzy_best_df["candidate_score"],
            errors="coerce"
        ).ge(threshold)
    ].copy()

    same_surname_count = int(
        as_bool(above_threshold["same_surname"]).sum()
    )
    same_surname_initial_count = int(
        (
            as_bool(above_threshold["same_surname"])
            & as_bool(above_threshold["first_initial_matches"])
        ).sum()
    )

    threshold_rows.append({
        "fuzzy_score_threshold": threshold,
        "benchmark_records_with_top_candidate_at_or_above_threshold": (
            len(above_threshold)
        ),
        "of_which_same_surname": same_surname_count,
        "of_which_same_surname_and_first_initial": (
            same_surname_initial_count
        ),
        "interpretation": (
            "Review workload diagnostic only; no fuzzy candidate is automatically counted as a match."
        )
    })

fuzzy_threshold_df = pd.DataFrame(threshold_rows)
write_csv_safely(fuzzy_threshold_df, FUZZY_THRESHOLD_PATH)

display(fuzzy_threshold_df)

## 4. Stratified manual-validation templates

Three review files are created and kept separate because they answer different quality questions.

| File | Purpose | Decision effect |
|---|---|---|
| `strict_match_manual_review_sample.csv` | Audit a reproducible stratified sample of strict automatic matches. | Does not automatically change the headline result. |
| `rule_based_match_manual_review.csv` | Review **all** initial–surname matches because this group is small and used only in the extended sensitivity result. | Can support a validated version of the extended sensitivity measure after all decisions are completed. |
| `fuzzy_candidate_manual_review_sample.csv` | Review a stratified sample of high/medium-priority fuzzy candidates. | Never changes primary coverage automatically. |

Use the official benchmark source and the linked OpenSanctions entity page to record the evidence supporting each decision. Keep the review reason concise and factual.

In [ ]:
# -------------------------------------------------------------------
# Create strict-match audit sample
# -------------------------------------------------------------------

strict_review_fields = [
    "manual_review_decision",
    "manual_review_reason",
    "manual_evidence_source",
    "manual_review_notes",
    "reviewer",
    "review_date"
]

strict_review_source_df = match_df.loc[
    strict_indicator
].copy()

strict_review_source_df["review_case_id"] = (
    "strict__"
    + strict_review_source_df["benchmark_record_id"].astype(str)
)

strict_review_source_df["opensanctions_entity_url"] = (
    "https://www.opensanctions.org/entities/"
    + strict_review_source_df.get(
        "opensanctions_id",
        pd.Series(pd.NA, index=strict_review_source_df.index)
    ).astype("string")
    + "/"
)

strict_sample_df = stratified_sample(
    strict_review_source_df,
    stratum_column="taxonomy_category",
    target_n=STRICT_AUDIT_SAMPLE_TARGET,
    random_seed=RANDOM_SEED
)

strict_template_columns = [
    "review_case_id",
    "benchmark_record_id",
    "person_name",
    "taxonomy_category",
    "role_title",
    "institution",
    "source_url",
    "match_type",
    "opensanctions_id",
    "opensanctions_name",
    "opensanctions_name_type",
    "countries",
    "dataset",
    "first_seen",
    "last_seen",
    "opensanctions_entity_url",
    *strict_review_fields
]

strict_sample_df = add_missing_columns(
    strict_sample_df,
    strict_template_columns
)[strict_template_columns].copy()

strict_sample_df = preserve_existing_review_fields(
    strict_sample_df,
    STRICT_REVIEW_PATH,
    key_column="review_case_id",
    review_columns=strict_review_fields
)

write_csv_safely(strict_sample_df, STRICT_REVIEW_PATH)

print(
    f"Strict-match audit sample: {len(strict_sample_df)} records "
    f"(target: {STRICT_AUDIT_SAMPLE_TARGET})."
)
display(strict_sample_df.head(20))

In [ ]:
# -------------------------------------------------------------------
# Create full rule-based review template
# -------------------------------------------------------------------

rule_review_fields = [
    "manual_review_decision",
    "manual_review_reason",
    "manual_evidence_source",
    "manual_review_notes",
    "reviewer",
    "review_date"
]

rule_review_source_df = match_df.loc[
    rule_indicator
].copy()

rule_review_source_df["review_case_id"] = (
    "rule__"
    + rule_review_source_df["benchmark_record_id"].astype(str)
)

rule_review_source_df["opensanctions_entity_url"] = (
    "https://www.opensanctions.org/entities/"
    + rule_review_source_df.get(
        "opensanctions_id",
        pd.Series(pd.NA, index=rule_review_source_df.index)
    ).astype("string")
    + "/"
)

rule_template_columns = [
    "review_case_id",
    "benchmark_record_id",
    "person_name",
    "taxonomy_category",
    "role_title",
    "institution",
    "source_url",
    "match_type",
    "opensanctions_id",
    "opensanctions_name",
    "countries",
    "dataset",
    "first_seen",
    "last_seen",
    "opensanctions_entity_url",
    *rule_review_fields
]

rule_review_df = add_missing_columns(
    rule_review_source_df,
    rule_template_columns
)[rule_template_columns].copy()

rule_review_df = preserve_existing_review_fields(
    rule_review_df,
    RULE_REVIEW_PATH,
    key_column="review_case_id",
    review_columns=rule_review_fields
)

write_csv_safely(rule_review_df, RULE_REVIEW_PATH)

print(
    f"Rule-based review template: {len(rule_review_df)} records. "
    "Review all records in this file before using the extended sensitivity result."
)
display(rule_review_df)

In [ ]:
# -------------------------------------------------------------------
# Create fuzzy-candidate manual-validation sample
# -------------------------------------------------------------------

fuzzy_review_fields = [
    "manual_review_decision",
    "manual_review_reason",
    "manual_evidence_source",
    "manual_review_notes",
    "reviewer",
    "review_date"
]

priority_values = {"high_priority_review", "medium_priority_review"}

fuzzy_review_source_df = fuzzy_candidates_df.loc[
    fuzzy_candidates_df["candidate_priority"].isin(priority_values)
].copy()

# Keep the best candidate for each unresolved benchmark record.
fuzzy_review_source_df = (
    fuzzy_review_source_df
    .sort_values(
        ["benchmark_record_id", "candidate_score", "candidate_rank"],
        ascending=[True, False, True],
        kind="stable"
    )
    .drop_duplicates(subset=["benchmark_record_id"], keep="first")
    .copy()
)

fuzzy_review_source_df["review_case_id"] = (
    "fuzzy__"
    + fuzzy_review_source_df["benchmark_record_id"].astype(str)
    + "__"
    + fuzzy_review_source_df["candidate_opensanctions_id"].astype(str)
)

fuzzy_review_source_df["opensanctions_entity_url"] = (
    "https://www.opensanctions.org/entities/"
    + fuzzy_review_source_df["candidate_opensanctions_id"].astype("string")
    + "/"
)

fuzzy_sample_df = stratified_sample(
    fuzzy_review_source_df,
    stratum_column="taxonomy_category",
    target_n=FUZZY_AUDIT_SAMPLE_TARGET,
    random_seed=RANDOM_SEED
)

fuzzy_template_columns = [
    "review_case_id",
    "benchmark_record_id",
    "person_name",
    "taxonomy_category",
    "role_title",
    "institution",
    "candidate_rank",
    "candidate_score",
    "same_surname",
    "first_initial_matches",
    "candidate_priority",
    "candidate_opensanctions_id",
    "candidate_opensanctions_name",
    "candidate_dataset",
    "opensanctions_entity_url",
    *fuzzy_review_fields
]

fuzzy_sample_df = add_missing_columns(
    fuzzy_sample_df,
    fuzzy_template_columns
)[fuzzy_template_columns].copy()

fuzzy_sample_df = preserve_existing_review_fields(
    fuzzy_sample_df,
    FUZZY_REVIEW_PATH,
    key_column="review_case_id",
    review_columns=fuzzy_review_fields
)

write_csv_safely(fuzzy_sample_df, FUZZY_REVIEW_PATH)

print(
    f"Fuzzy-candidate audit sample: {len(fuzzy_sample_df)} records "
    f"(target: {FUZZY_AUDIT_SAMPLE_TARGET})."
)
display(fuzzy_sample_df.head(20))

### Review instructions

For the strict and rule-based files, use one of:

- `confirmed_correct_match`
- `false_positive`
- `uncertain`
- `not_reviewed`

For the fuzzy file, use one of:

- `confirmed_match`
- `not_a_match`
- `insufficient_evidence`
- `not_reviewed`

The standardised reason can be, for example: `same_person_role_context_confirmed`, `different_person`, `name_variant_but_role_not_confirmed`, `insufficient_public_evidence`, or `source_date_mismatch`.

After completing and saving the CSV files in Excel, close them and rerun the next section. The notebook will preserve your decisions.

In [ ]:
# -------------------------------------------------------------------
# Summarise completed manual-review decisions
# -------------------------------------------------------------------

def normalise_decision(review_type: str, value) -> str:
    """Map review-specific labels to a small common set for summaries."""
    if pd.isna(value):
        return "not_reviewed"

    decision = str(value).strip().lower()

    if decision in {"", "not_reviewed"}:
        return "not_reviewed"

    if review_type in {"strict", "rule_based"}:
        if decision == "confirmed_correct_match":
            return "match"
        if decision == "false_positive":
            return "non_match"
        if decision == "uncertain":
            return "uncertain"

    if review_type == "fuzzy":
        if decision == "confirmed_match":
            return "match"
        if decision == "not_a_match":
            return "non_match"
        if decision == "insufficient_evidence":
            return "uncertain"

    return "other_or_invalid_value"


def prepare_review_summary_source(
    dataframe: pd.DataFrame,
    review_type: str,
    decision_column: str = "manual_review_decision"
) -> pd.DataFrame:
    summary_df = dataframe.copy()
    summary_df["review_type"] = review_type
    summary_df["review_decision_standardised"] = summary_df[
        decision_column
    ].apply(lambda value: normalise_decision(review_type, value))
    return summary_df


strict_review_summary_source_df = prepare_review_summary_source(
    strict_sample_df,
    "strict"
)
rule_review_summary_source_df = prepare_review_summary_source(
    rule_review_df,
    "rule_based"
)
fuzzy_review_summary_source_df = prepare_review_summary_source(
    fuzzy_sample_df,
    "fuzzy"
)

all_review_df = pd.concat(
    [
        strict_review_summary_source_df,
        rule_review_summary_source_df,
        fuzzy_review_summary_source_df
    ],
    ignore_index=True,
    sort=False
)

manual_review_summary_df = (
    all_review_df
    .groupby(
        ["review_type", "review_decision_standardised"],
        dropna=False
    )
    .size()
    .reset_index(name="review_cases")
    .sort_values(["review_type", "review_decision_standardised"])
    .reset_index(drop=True)
)

reviewed_strict_false_positives = int(
    (
        strict_review_summary_source_df["review_decision_standardised"]
        .eq("non_match")
    ).sum()
)
reviewed_rule_false_positives = int(
    (
        rule_review_summary_source_df["review_decision_standardised"]
        .eq("non_match")
    ).sum()
)
fuzzy_non_matches = int(
    (
        fuzzy_review_summary_source_df["review_decision_standardised"]
        .eq("non_match")
    ).sum()
)
fuzzy_confirmed_matches = int(
    (
        fuzzy_review_summary_source_df["review_decision_standardised"]
        .eq("match")
    ).sum()
)

matching_quality_findings_df = pd.DataFrame([
    {
        "finding": "strict_match_audit_sample_size",
        "value": len(strict_sample_df),
        "interpretation": "Reproducible stratified audit sample; not a population accuracy estimate."
    },
    {
        "finding": "completed_strict_match_reviews",
        "value": int(
            strict_review_summary_source_df[
                "review_decision_standardised"
            ].ne("not_reviewed").sum()
        ),
        "interpretation": "Strict sample records with a manual decision."
    },
    {
        "finding": "strict_sample_false_positive_findings",
        "value": reviewed_strict_false_positives,
        "interpretation": "Observed false positives in the reviewed strict sample only; do not extrapolate automatically."
    },
    {
        "finding": "rule_based_matches_requiring_audit",
        "value": len(rule_review_df),
        "interpretation": "All initial–surname matches should be reviewed before relying on extended coverage."
    },
    {
        "finding": "rule_based_false_positive_findings",
        "value": reviewed_rule_false_positives,
        "interpretation": "Observed rejected initial–surname matches."
    },
    {
        "finding": "fuzzy_candidate_sample_size",
        "value": len(fuzzy_sample_df),
        "interpretation": "Reproducible stratified sample of high/medium-priority fuzzy candidates."
    },
    {
        "finding": "fuzzy_candidate_non_match_findings",
        "value": fuzzy_non_matches,
        "interpretation": "Candidate-generation false positives in the reviewed fuzzy sample."
    },
    {
        "finding": "manually_confirmed_fuzzy_candidates",
        "value": fuzzy_confirmed_matches,
        "interpretation": (
            "Potential strict-method misses found in the reviewed sample. "
            "They are not added to primary coverage unless the matching protocol is formally amended."
        )
    }
])

write_csv_safely(manual_review_summary_df, REVIEW_SUMMARY_PATH)
write_csv_safely(matching_quality_findings_df, MATCH_FINDINGS_PATH)

print("Manual-review summary:")
display(manual_review_summary_df)

print("\nMatching-quality findings:")
display(matching_quality_findings_df)

## 5. Post-review matching audit and final sensitivity outputs

This section uses the completed manual-review files. It distinguishes the primary strict estimate from pre-review candidate counts. The **post-review sensitivity** is the figure to report if manually confirmed initial–surname links are discussed. Fuzzy confirmations remain a diagnostic finding and are not incorporated into the pre-specified primary estimate.

In [ ]:
# -------------------------------------------------------------------
# Post-review matching audit and final sensitivity outputs
# -------------------------------------------------------------------

# This section must be run only after the completed review CSV files have been
# placed in data/output/validation_checks/. The notebook preserves those review
# fields by review_case_id when rerun.

strict_manual_match_count = int(
    strict_review_summary_source_df["review_decision_standardised"].eq("match").sum()
)
strict_manual_non_match_count = int(
    strict_review_summary_source_df["review_decision_standardised"].eq("non_match").sum()
)

rule_manual_match_count = int(
    rule_review_summary_source_df["review_decision_standardised"].eq("match").sum()
)
rule_manual_non_match_count = int(
    rule_review_summary_source_df["review_decision_standardised"].eq("non_match").sum()
)

fuzzy_manual_match_count = int(
    fuzzy_review_summary_source_df["review_decision_standardised"].eq("match").sum()
)
fuzzy_manual_non_match_count = int(
    fuzzy_review_summary_source_df["review_decision_standardised"].eq("non_match").sum()
)

manual_confirmed_sensitivity_count = strict_count + rule_manual_match_count
exploratory_manual_linkage_count = manual_confirmed_sensitivity_count + fuzzy_manual_match_count

post_review_audit_df = pd.DataFrame([
    {
        "review_group": "strict_confirmed_matches",
        "selection_design": "Reproducible stratified audit sample across taxonomy categories",
        "reviewed_cases": len(strict_sample_df),
        "confirmed_matches": strict_manual_match_count,
        "rejected_non_matches": strict_manual_non_match_count,
        "interpretation": "Audit evidence for the strict procedure; not a population precision estimate."
    },
    {
        "review_group": "initial_surname_candidates",
        "selection_design": "Census of all unique initial-surname candidate links",
        "reviewed_cases": len(rule_review_df),
        "confirmed_matches": rule_manual_match_count,
        "rejected_non_matches": rule_manual_non_match_count,
        "interpretation": "Only manually confirmed candidates may be discussed as a secondary sensitivity result."
    },
    {
        "review_group": "high_medium_priority_fuzzy_candidates",
        "selection_design": "Census of all high- and medium-priority candidates",
        "reviewed_cases": len(fuzzy_sample_df),
        "confirmed_matches": fuzzy_manual_match_count,
        "rejected_non_matches": fuzzy_manual_non_match_count,
        "interpretation": "Diagnostic evidence of potential strict-method misses; not added to the pre-specified primary estimate."
    }
])

post_review_sensitivity_df = pd.DataFrame([
    {
        "scenario": "primary_strict_confirmed_coverage",
        "numerator_matches": strict_count,
        "denominator_records": benchmark_total,
        "coverage_rate": strict_count / benchmark_total if benchmark_total else np.nan,
        "interpretation": "Primary result: unique normalised exact and cleaned exact matches."
    },
    {
        "scenario": "strict_plus_manually_confirmed_initial_surname_sensitivity",
        "numerator_matches": manual_confirmed_sensitivity_count,
        "denominator_records": benchmark_total,
        "coverage_rate": (
            manual_confirmed_sensitivity_count / benchmark_total
            if benchmark_total else np.nan
        ),
        "interpretation": (
            "Secondary sensitivity: strict matches plus only those initial-surname candidates "
            "confirmed through manual review."
        )
    },
    {
        "scenario": "strict_coverage_excluding_ambiguous_exact_name_records",
        "numerator_matches": strict_count,
        "denominator_records": non_ambiguous_denominator,
        "coverage_rate": (
            strict_count / non_ambiguous_denominator
            if non_ambiguous_denominator else np.nan
        ),
        "interpretation": (
            "Denominator sensitivity only: removes exact-name collision cases; does not "
            "change the matching protocol."
        )
    },
    {
        "scenario": "exploratory_strict_plus_all_manually_confirmed_non_strict_links",
        "numerator_matches": exploratory_manual_linkage_count,
        "denominator_records": benchmark_total,
        "coverage_rate": (
            exploratory_manual_linkage_count / benchmark_total
            if benchmark_total else np.nan
        ),
        "interpretation": (
            "Exploratory diagnostic only: adds manually confirmed initial-surname and fuzzy "
            "links. Do not use as the headline measure because fuzzy confirmation occurred "
            "after the pre-specified deterministic protocol."
        )
    }
])
post_review_sensitivity_df["coverage_percentage"] = (
    post_review_sensitivity_df["coverage_rate"] * 100
).round(1)

write_csv_safely(post_review_audit_df, POST_REVIEW_AUDIT_PATH)
write_csv_safely(post_review_sensitivity_df, POST_REVIEW_SENSITIVITY_PATH)

print("Post-review matching audit:")
display(post_review_audit_df)

print("\nPost-review sensitivity analysis:")
display(post_review_sensitivity_df)


## 6. Intra-coder consistency check

Complete the initial review files first. Then rerun this section to create a **blinded repeat-review template** containing a reproducible subset of completed cases.

For a meaningful check, complete the repeat review later and do not consult the original decision while recoding. The output reports simple percent agreement for paired match/non-match decisions. It is deliberately reported as **intra-coder consistency**, not inter-rater reliability.

In [ ]:
# -------------------------------------------------------------------
# Prepare eligible initial decisions for repeat review
# -------------------------------------------------------------------

review_case_columns = [
    "review_case_id",
    "review_type",
    "benchmark_record_id",
    "taxonomy_category",
    "person_name",
    "role_title",
    "institution",
    "opensanctions_entity_url",
    "manual_review_decision",
    "review_decision_standardised"
]

repeat_source_df = add_missing_columns(
    all_review_df,
    review_case_columns
)[review_case_columns].copy()

# Add the OpenSanctions display name, using the relevant field for each review type.
repeat_source_df["opensanctions_name_for_review"] = pd.NA

strict_rule_mask = repeat_source_df["review_type"].isin(
    ["strict", "rule_based"]
)
fuzzy_mask = repeat_source_df["review_type"].eq("fuzzy")

strict_rule_lookup = pd.concat(
    [
        strict_sample_df[
            ["review_case_id", "opensanctions_id", "opensanctions_name"]
        ],
        rule_review_df[
            ["review_case_id", "opensanctions_id", "opensanctions_name"]
        ]
    ],
    ignore_index=True
).drop_duplicates(subset=["review_case_id"], keep="last")

fuzzy_lookup = fuzzy_sample_df[
    [
        "review_case_id",
        "candidate_opensanctions_id",
        "candidate_opensanctions_name",
        "candidate_score"
    ]
].rename(
    columns={
        "candidate_opensanctions_id": "opensanctions_id",
        "candidate_opensanctions_name": "opensanctions_name"
    }
)

repeat_source_df = (
    repeat_source_df
    .drop(columns=["opensanctions_name_for_review"], errors="ignore")
    .merge(
        pd.concat(
            [
                strict_rule_lookup,
                fuzzy_lookup
            ],
            ignore_index=True,
            sort=False
        ).drop_duplicates(subset=["review_case_id"], keep="last"),
        on="review_case_id",
        how="left"
    )
)

repeat_source_df["review_stratum"] = (
    repeat_source_df["review_type"].astype(str)
    + " | "
    + repeat_source_df["taxonomy_category"].astype(str)
)

eligible_repeat_source_df = repeat_source_df.loc[
    repeat_source_df["review_decision_standardised"].isin(
        ["match", "non_match"]
    )
].copy()

repeat_review_fields = [
    "repeat_review_decision",
    "repeat_review_reason",
    "repeat_evidence_source",
    "repeat_review_notes",
    "repeat_reviewer",
    "repeat_review_date"
]

repeat_template_columns = [
    "review_case_id",
    "review_type",
    "benchmark_record_id",
    "taxonomy_category",
    "person_name",
    "role_title",
    "institution",
    "opensanctions_id",
    "opensanctions_name",
    "opensanctions_entity_url",
    "candidate_score",
    *repeat_review_fields
]

existing_repeat_review_df = pd.DataFrame()

if REPEAT_REVIEW_PATH.exists():
    existing_repeat_review_df = read_csv_with_fallback_encodings(
        REPEAT_REVIEW_PATH,
        required=False
    )

if not existing_repeat_review_df.empty:
    repeat_review_df = add_missing_columns(
        existing_repeat_review_df,
        repeat_template_columns
    )[repeat_template_columns].copy()

    print(
        "Existing repeat-review template reloaded. "
        "The original selected cases and any completed repeat decisions were preserved."
    )

elif eligible_repeat_source_df.empty:
    repeat_review_df = pd.DataFrame(columns=repeat_template_columns)

    print(
        "No repeat-review template was created yet because no completed initial "
        "match/non-match decisions were found. Complete the manual-review files "
        "first, save them, and rerun this section."
    )

else:
    repeat_sample_source_df = stratified_sample(
        eligible_repeat_source_df,
        stratum_column="review_stratum",
        target_n=REPEAT_REVIEW_SAMPLE_TARGET,
        random_seed=RANDOM_SEED
    )

    # The initial decision is deliberately excluded from the exported repeat file.
    repeat_review_df = add_missing_columns(
        repeat_sample_source_df,
        repeat_template_columns
    )[repeat_template_columns].copy()

    write_csv_safely(repeat_review_df, REPEAT_REVIEW_PATH)

    print(
        f"Created a blinded repeat-review template with {len(repeat_review_df)} cases. "
        "Complete it later without consulting the original decision."
    )

if not REPEAT_REVIEW_PATH.exists():
    write_csv_safely(repeat_review_df, REPEAT_REVIEW_PATH)

display(repeat_review_df.head(20))

In [ ]:
# -------------------------------------------------------------------
# Calculate intra-coder agreement after repeat review
# -------------------------------------------------------------------

repeat_review_df = read_csv_with_fallback_encodings(
    REPEAT_REVIEW_PATH,
    required=False
)

repeat_review_df = add_missing_columns(
    repeat_review_df,
    [
        "review_case_id",
        "repeat_review_decision",
        "review_type",
        "taxonomy_category",
        "person_name"
    ]
)

def normalise_repeat_decision(review_type: str, value) -> str:
    """Use the same decision mapping as the initial review."""
    return normalise_decision(review_type, value)

repeat_review_df["repeat_decision_standardised"] = repeat_review_df.apply(
    lambda row: normalise_repeat_decision(
        row["review_type"],
        row["repeat_review_decision"]
    ),
    axis=1
)

initial_decision_lookup_df = repeat_source_df[
    [
        "review_case_id",
        "review_type",
        "review_decision_standardised"
    ]
].drop_duplicates(subset=["review_case_id"], keep="last")

intra_coder_pairs_df = repeat_review_df.merge(
    initial_decision_lookup_df,
    on=["review_case_id", "review_type"],
    how="left"
)

paired_binary_decisions_df = intra_coder_pairs_df.loc[
    intra_coder_pairs_df["review_decision_standardised"].isin(
        ["match", "non_match"]
    )
    & intra_coder_pairs_df["repeat_decision_standardised"].isin(
        ["match", "non_match"]
    )
].copy()

paired_binary_decisions_df["same_decision"] = (
    paired_binary_decisions_df["review_decision_standardised"]
    .eq(paired_binary_decisions_df["repeat_decision_standardised"])
)

overall_pairs = len(paired_binary_decisions_df)
overall_agreements = int(
    paired_binary_decisions_df["same_decision"].sum()
) if overall_pairs else 0

intra_coder_summary_rows = [
    {
        "review_group": "overall",
        "paired_binary_decisions": overall_pairs,
        "agreements": overall_agreements,
        "percent_agreement": (
            round(overall_agreements / overall_pairs * 100, 1)
            if overall_pairs else np.nan
        ),
        "interpretation": (
            "Intra-coder consistency only; no second independent reviewer was used."
        )
    }
]

for review_type, subset_df in paired_binary_decisions_df.groupby(
    "review_type",
    dropna=False
):
    pair_count = len(subset_df)
    agreement_count = int(subset_df["same_decision"].sum())

    intra_coder_summary_rows.append({
        "review_group": review_type,
        "paired_binary_decisions": pair_count,
        "agreements": agreement_count,
        "percent_agreement": (
            round(agreement_count / pair_count * 100, 1)
            if pair_count else np.nan
        ),
        "interpretation": "Agreement between first and blinded repeat review."
    })

intra_coder_summary_df = pd.DataFrame(intra_coder_summary_rows)

intra_coder_disagreements_df = paired_binary_decisions_df.loc[
    ~paired_binary_decisions_df["same_decision"]
].copy()

write_csv_safely(intra_coder_summary_df, INTRA_CODER_SUMMARY_PATH)
write_csv_safely(
    intra_coder_disagreements_df,
    INTRA_CODER_DISAGREEMENTS_PATH
)

print("Intra-coder consistency summary:")
display(intra_coder_summary_df)

print("\nDisagreements requiring explanation:")
display(intra_coder_disagreements_df)

## 7. Reproducibility record

This section saves:

- a file manifest with SHA-256 fingerprints for the primary project inputs and analysis outputs;
- the Python and package versions used to run the notebook;
- configured seeds and review-sample sizes; and
- row counts at the main pipeline stages.

Keep these files with the final repository or submission appendix. The OpenSanctions export can be large, so its fingerprint may take longer to calculate than the other files.

In [ ]:
# -------------------------------------------------------------------
# Reproducibility manifest and environment record
# -------------------------------------------------------------------

manifest_files = {
    "input__pep_source_mapping": INPUT_DIR / "PEP_source_mapping.csv",
    "input__taxonomy": INPUT_DIR / "taxonomy_v2_eu_aligned.xlsx",
    "input__manual_main_additions": INPUT_DIR / "manual_main_v2_additions.xlsx",
    "input__manual_small_additions": INPUT_DIR / "manual_small_official_additions.csv",
    "input__party_board_manual": INPUT_DIR / "category_c_party_board_manual_v3.csv",
    "input__party_sources": INPUT_DIR / "category_c_party_sources.csv",
    "external__opensanctions_export": OPEN_SANCTIONS_PATH,
    "output__benchmark_before_cleaning": RAW_COMPONENTS_PATH,
    "output__final_benchmark": BENCHMARK_PATH,
    "output__final_match_results": MATCH_RESULTS_PATH,
    "output__match_summary": MATCH_SUMMARY_PATH,
    "output__fuzzy_candidates": FUZZY_CANDIDATES_PATH,
    "output__rule_based_matches": RULE_BASED_MATCHES_PATH,
}

manifest_rows = []

for file_role, file_path in manifest_files.items():
    if file_path.exists():
        stat = file_path.stat()

        manifest_rows.append({
            "file_role": file_role,
            "path_relative_to_project": str(
                file_path.relative_to(PROJECT_DIR)
            ),
            "exists": True,
            "size_bytes": stat.st_size,
            "modified_utc": datetime.fromtimestamp(
                stat.st_mtime,
                tz=timezone.utc
            ).isoformat(),
            "sha256": file_sha256(file_path)
        })
    else:
        manifest_rows.append({
            "file_role": file_role,
            "path_relative_to_project": str(
                file_path.relative_to(PROJECT_DIR)
            ),
            "exists": False,
            "size_bytes": pd.NA,
            "modified_utc": pd.NA,
            "sha256": pd.NA
        })

reproducibility_manifest_df = pd.DataFrame(manifest_rows)
write_csv_safely(reproducibility_manifest_df, MANIFEST_PATH)

environment_df = pd.DataFrame([
    {
        "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
        "python_version": sys.version.replace("\n", " "),
        "python_executable": sys.executable,
        "operating_system": platform.platform(),
        "pandas_version": package_version("pandas"),
        "numpy_version": package_version("numpy"),
        "rapidfuzz_version": package_version("RapidFuzz"),
        "openpyxl_version": package_version("openpyxl")
    }
])

write_csv_safely(environment_df, ENVIRONMENT_PATH)

validation_parameters = {
    "random_seed": RANDOM_SEED,
    "strict_audit_sample_target": STRICT_AUDIT_SAMPLE_TARGET,
    "fuzzy_audit_sample_target": FUZZY_AUDIT_SAMPLE_TARGET,
    "repeat_review_sample_target": REPEAT_REVIEW_SAMPLE_TARGET,
    "strict_match_types": sorted(strict_match_types),
    "rule_based_match_types": sorted(rule_match_types),
    "fuzzy_thresholds_diagnostic_only": [85, 90, 95],
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat()
}

RUN_METADATA_PATH.write_text(
    json.dumps(validation_parameters, indent=2),
    encoding="utf-8"
)

print("Reproducibility manifest:")
display(reproducibility_manifest_df)

print("\nPython environment record:")
display(environment_df)

In [ ]:
# -------------------------------------------------------------------
# Pipeline row-count record
# -------------------------------------------------------------------

pipeline_row_counts_df = pd.DataFrame([
    {
        "pipeline_stage": "components_before_cleaning",
        "records": len(raw_components_df),
        "source_file": str(RAW_COMPONENTS_PATH.relative_to(PROJECT_DIR))
    },
    {
        "pipeline_stage": "final_benchmark",
        "records": len(benchmark_df),
        "source_file": str(BENCHMARK_PATH.relative_to(PROJECT_DIR))
    },
    {
        "pipeline_stage": "final_match_results",
        "records": len(match_df),
        "source_file": str(MATCH_RESULTS_PATH.relative_to(PROJECT_DIR))
    },
    {
        "pipeline_stage": "strict_confirmed_matches",
        "records": strict_count,
        "source_file": str(MATCH_RESULTS_PATH.relative_to(PROJECT_DIR))
    },
    {
        "pipeline_stage": "rule_based_initial_surname_matches",
        "records": rule_based_count,
        "source_file": str(MATCH_RESULTS_PATH.relative_to(PROJECT_DIR))
    },
    {
        "pipeline_stage": "pre_review_initial_surname_candidate_links",
        "records": rule_based_count,
        "source_file": str(MATCH_RESULTS_PATH.relative_to(PROJECT_DIR))
    },
    {
        "pipeline_stage": "manually_confirmed_initial_surname_matches",
        "records": rule_manual_match_count,
        "source_file": str(RULE_REVIEW_PATH.relative_to(PROJECT_DIR))
    },
    {
        "pipeline_stage": "strict_plus_manually_confirmed_initial_surname_matches",
        "records": manual_confirmed_sensitivity_count,
        "source_file": str(POST_REVIEW_SENSITIVITY_PATH.relative_to(PROJECT_DIR))
    },
    {
        "pipeline_stage": "fuzzy_candidate_rows",
        "records": len(fuzzy_candidates_df),
        "source_file": str(FUZZY_CANDIDATES_PATH.relative_to(PROJECT_DIR))
    },
    {
        "pipeline_stage": "fuzzy_records_in_manual_sample",
        "records": len(fuzzy_sample_df),
        "source_file": str(FUZZY_REVIEW_PATH.relative_to(PROJECT_DIR))
    }
])

write_csv_safely(pipeline_row_counts_df, ROW_COUNTS_PATH)

print("Pipeline row-count record:")
display(pipeline_row_counts_df)

## 8. Final validation dashboard

Use this final overview to identify what still needs attention before reporting results.

A `review` result is not automatically a failure. It is an item requiring one of three defensible actions: correct the data, provide documented evidence for the exception, or state the limitation transparently in the thesis.

In [ ]:
# -------------------------------------------------------------------
# Final dashboard
# -------------------------------------------------------------------

dashboard_rows = [
    {
        "area": "dataset_integrity",
        "checks_requiring_review": int(
            dataset_integrity_df["status"].ne("pass").sum()
        ),
        "main_output": DATASET_INTEGRITY_PATH.name
    },
    {
        "area": "benchmark_quality",
        "checks_requiring_review": int(
            benchmark_quality_df["status"].ne("pass").sum()
        ),
        "main_output": BENCHMARK_QUALITY_PATH.name
    },
    {
        "area": "matching_integrity",
        "checks_requiring_review": int(
            matching_integrity_df["status"].ne("pass").sum()
        ),
        "main_output": MATCHING_INTEGRITY_PATH.name
    },
    {
        "area": "strict_manual_review",
        "checks_requiring_review": int(
            strict_review_summary_source_df[
                "review_decision_standardised"
            ].eq("not_reviewed").sum()
        ),
        "main_output": STRICT_REVIEW_PATH.name
    },
    {
        "area": "rule_based_manual_review",
        "checks_requiring_review": int(
            rule_review_summary_source_df[
                "review_decision_standardised"
            ].eq("not_reviewed").sum()
        ),
        "main_output": RULE_REVIEW_PATH.name
    },
    {
        "area": "fuzzy_manual_review",
        "checks_requiring_review": int(
            fuzzy_review_summary_source_df[
                "review_decision_standardised"
            ].eq("not_reviewed").sum()
        ),
        "main_output": FUZZY_REVIEW_PATH.name
    },
    {
        "area": "intra_coder_repeat_review",
        "checks_requiring_review": int(
            repeat_review_df["repeat_review_decision"]
            .astype("string")
            .fillna("")
            .str.strip()
            .isin(["", "not_reviewed"])
            .sum()
        ) if not repeat_review_df.empty else 0,
        "main_output": REPEAT_REVIEW_PATH.name
    }
]

validation_dashboard_df = pd.DataFrame(dashboard_rows)

print("Validation dashboard:")
display(validation_dashboard_df)

print("\nPrimary outputs saved in:")
print(VALIDATION_DIR)

print(
    "\nRecommended sequence:\n"
    "1. Resolve or document benchmark-quality exceptions.\n"
    "2. Complete the strict, rule-based, and fuzzy review templates.\n"
    "3. Rerun this notebook to update manual-review summaries.\n"
    "4. Complete the blinded repeat-review template later and rerun once more.\n"
    "5. Use post_review_sensitivity_analysis.csv for thesis reporting: strict is primary; manually confirmed initial-surname links are a secondary sensitivity result."
)